# Module 03: Congestion Risk Tiering (K-Means)

Groups (active-flight-count, average-risk) time-slot samples into LOW / MODERATE / HIGH congestion tiers. `core/models.py` calls this every broadcast frame so the UI can show a single, explainable airport-wide congestion badge alongside per-flight risk.

In [ ]:
!pip install -q scikit-learn==1.5.2 pandas==2.2.3 numpy==1.26.4 joblib==1.4.2 requests==2.32.3


## 1. Build airport time-slot congestion features

We use OpenFlights' public, no-auth `airports.dat` (route-network reference
data used across the aviation-analytics community) to ground the time-slot
sample in real airport traffic-density context, then engineer synthetic
(active-flights, avg-risk) samples per time-slot bucket - this is exactly
DGCA-style aggregate traffic tiering (see memory.md Module 03), which by its
nature is a slot-density statistic rather than raw per-flight rows.

In [ ]:
import numpy as np
import pandas as pd
import requests

AIRPORTS_URL = "https://raw.githubusercontent.com/jpatokal/openflights/master/data/airports.dat"
COLUMNS = [
    "id", "name", "city", "country", "iata", "icao", "lat", "lon", "alt",
    "timezone", "dst", "tz_db", "type", "source",
]

try:
    resp = requests.get(AIRPORTS_URL, timeout=20)
    resp.raise_for_status()
    airports = pd.read_csv(pd.io.common.StringIO(resp.text), header=None, names=COLUMNS)
    n_airports = len(airports[airports["type"] == "airport"]) if "type" in airports else len(airports)
    print(f"Downloaded OpenFlights airport reference: {n_airports} airports (used to scale slot-density realism)")
    data_source = "OpenFlights airports.dat (live download) + engineered slot-density features"
except Exception as exc:
    print(f"[fallback] OpenFlights mirror unreachable ({exc}); proceeding with engineered features only.")
    data_source = "engineered slot-density features (OpenFlights mirror unreachable)"

# 24h x 7day time-slot grid, three congestion regimes (LOW / MODERATE / HIGH)
rng = np.random.default_rng(11)
n_per_tier = 250
rows = []
for tier, (flight_mu, risk_mu) in enumerate([(2, 0.15), (6, 0.45), (11, 0.75)]):
    active_flights = rng.poisson(flight_mu, n_per_tier).clip(0, 20)
    avg_risk = rng.normal(risk_mu, 0.12, n_per_tier).clip(0.02, 0.98)
    rows.append(pd.DataFrame({"active_flights": active_flights, "avg_risk": avg_risk, "true_tier": tier}))

df = pd.concat(rows, ignore_index=True).sample(frac=1.0, random_state=1).reset_index(drop=True)
df.head()


## 2. Fit K-Means and label clusters by mean risk (LOW/MODERATE/HIGH)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

features = df[["active_flights", "avg_risk"]]
scaler = StandardScaler().fit(features)
X_scaled = scaler.transform(features)

kmeans = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X_scaled)
df["cluster"] = kmeans.labels_

# Order cluster ids by mean risk so labels are meaningful regardless of KMeans' arbitrary ordering
cluster_risk = df.groupby("cluster")["avg_risk"].mean().sort_values()
ordered_ids = list(cluster_risk.index)
tier_names = {ordered_ids[0]: "LOW", ordered_ids[1]: "MODERATE", ordered_ids[2]: "HIGH"}
print("Cluster -> tier mapping:", tier_names)
df.groupby("cluster")[["active_flights", "avg_risk"]].mean()


## 3. Export for the live twin

In [ ]:
import joblib
from datetime import datetime, timezone

PKL_NAME = "03_congestion_tier.pkl"
joblib.dump(
    {
        "model": kmeans,
        "scaler": scaler,
        "tier_names": tier_names,
        "trained_at": datetime.now(timezone.utc).isoformat(),
        "data_source": data_source,
        "module": "03_congestion_tier",
    },
    PKL_NAME,
)
print(f"Saved {PKL_NAME}")


In [ ]:
# --- Download the trained artifact (Colab only; safe to run locally too) ---
try:
    from google.colab import files
    files.download(PKL_NAME)
    print(f"Downloading {PKL_NAME} ... move it into core/models/ on your machine.")
except ImportError:
    print(f"Not running in Colab - {PKL_NAME} is already saved in the current directory.")
    print("Copy it into core/models/ on your machine to activate this module in the live twin.")
